1. The data is provided in csv files: purchases2021.csv, purchases2022.csv, purchases2023.csv, purchases2024.csv. Read the files using PySpark. Combine them into one data frame by adding a year column (according to the file name).
2. As it turned out, the data does not match the structure of the main MongoDB collection and cannot be imported as is. Find duplicate purchases (same id). Among purchases with different cost and the same id, select the maximum one.
3. The user IDs used in one of the offices have been simplified and now do not match the IDs in the main database. Use the customers.csv file to convert the user IDs to the correct format.
4. Also, the reviews for purchases were exported to a separate file reviews.csv. In the MongoDB database, the review is a field of the purchases collection. Convert the review as follows.
5. Select only the columns you need. Rename the columns similarly to the example of processing customer data.
6. In PySpark, determine the ids of 5 WIP customers – customers who bought products for the largest amount.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.functions import sum as _sum, col

In [2]:
spark = SparkSession.builder.appName("Application")
spark = spark.config("spark.mongodb.read.connection.uri", "mongodb://127.0.0.1:27017/shopdb")
spark = spark.config("spark.mongodb.write.connection.uri", "mongodb://127.0.0.1:27017/shopdb")
spark = spark.config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:10.4.0")
spark = spark.getOrCreate()

In [3]:
purchases2021 = spark.read.csv('purchases_2021.csv', header=True, inferSchema=True)
purchases2022 = spark.read.csv('purchases_2022.csv', header=True, inferSchema=True)
purchases2023 = spark.read.csv('purchases_2023.csv', header=True, inferSchema=True)
purchases2024 = spark.read.csv('purchases_2024.csv', header=True, inferSchema=True)

In [4]:
purchases2021 = purchases2021.withColumn('year', lit(2021))
purchases2022 = purchases2022.withColumn('year', lit(2022))
purchases2023 = purchases2023.withColumn('year', lit(2023))
purchases2024 = purchases2024.withColumn('year', lit(2024))

In [5]:
purchases = purchases2021.unionByName(purchases2022).unionByName(purchases2023).unionByName(purchases2024)

In [6]:
categories_df = purchases.select('id', 'category').dropDuplicates(['id'])
purchases = purchases.groupBy(purchases.id).max()
purchases = purchases.withColumnRenamed('max(customer_id)', 'customer_id').withColumnRenamed('max(cost)', 'cost').withColumnRenamed('max(year)', 'year').drop('max(id)')

In [7]:
purchases = purchases.join(categories_df, on='id', how='left')

In [8]:
purchases.show(5)

+---+----+-----------+----+------------+
| id|cost|customer_id|year|    category|
+---+----+-----------+----+------------+
|251|8904|         89|2021| Electronics|
| 81|6741|        745|2021|       Books|
|847|3290|        917|2021|Toys & Games|
|787|5709|        173|2021|        Food|
| 93|5988|        183|2021|       Books|
+---+----+-----------+----+------------+
only showing top 5 rows


In [9]:
customers = spark.read.csv('customers.csv', header=True, inferSchema=True)
customers.show(5)

+-----------+------------+
|customer_id|  correct_id|
+-----------+------------+
|          0|540klx2qsemc|
|          1|t704mot7fsm3|
|          2|15b0tblir2rp|
|          3|7jh0rkzjnypp|
|          4|ne3pefzqqzxc|
+-----------+------------+
only showing top 5 rows


In [10]:
purchases = purchases.join(customers, on='customer_id', how='left')
purchases.show(5)

+-----------+---+----+----+------------+------------+
|customer_id| id|cost|year|    category|  correct_id|
+-----------+---+----+----+------------+------------+
|         89|251|8904|2021| Electronics|gj75pex6zw54|
|        745| 81|6741|2021|       Books|9llf7yv6ugsa|
|        917|847|3290|2021|Toys & Games|83ew5590inb9|
|        173|787|5709|2021|        Food|z5p6aqq2de27|
|        183| 93|5988|2021|       Books|zqeo3nr0zqez|
+-----------+---+----+----+------------+------------+
only showing top 5 rows


In [11]:
purchases = purchases.drop('customer_id').withColumnRenamed('correct_id', 'customer_id')
purchases.show(5)

+---+----+----+------------+------------+
| id|cost|year|    category| customer_id|
+---+----+----+------------+------------+
|251|8904|2021| Electronics|gj75pex6zw54|
| 81|6741|2021|       Books|9llf7yv6ugsa|
|847|3290|2021|Toys & Games|83ew5590inb9|
|787|5709|2021|        Food|z5p6aqq2de27|
| 93|5988|2021|       Books|zqeo3nr0zqez|
+---+----+----+------------+------------+
only showing top 5 rows


In [12]:
reviews = spark.read.csv('reviews.csv', header=True, inferSchema=True)
purchases = purchases.join(reviews, purchases.id == reviews.purchase_id, how='left')
purchases = purchases.drop('purchase_id')
purchases.show(5)

+---+----+----+------------+------------+--------------------+
| id|cost|year|    category| customer_id|              review|
+---+----+----+------------+------------+--------------------+
|251|8904|2021| Electronics|gj75pex6zw54|                NULL|
| 81|6741|2021|       Books|9llf7yv6ugsa|                NULL|
|847|3290|2021|Toys & Games|83ew5590inb9|This air fryer is...|
|787|5709|2021|        Food|z5p6aqq2de27|                NULL|
| 93|5988|2021|       Books|zqeo3nr0zqez|                NULL|
+---+----+----+------------+------------+--------------------+
only showing top 5 rows


In [16]:
purchases = purchases.withColumnRenamed('id', '_id')
purchases = purchases.select('_id', 'cost', 'customer_id', 'category', 'year', 'review')
purchases.show(5)

+---+----+------------+------------+----+--------------------+
|_id|cost| customer_id|    category|year|              review|
+---+----+------------+------------+----+--------------------+
|251|8904|gj75pex6zw54| Electronics|2021|                NULL|
| 81|6741|9llf7yv6ugsa|       Books|2021|                NULL|
|847|3290|83ew5590inb9|Toys & Games|2021|This air fryer is...|
|787|5709|z5p6aqq2de27|        Food|2021|                NULL|
| 93|5988|zqeo3nr0zqez|       Books|2021|                NULL|
+---+----+------------+------------+----+--------------------+
only showing top 5 rows


In [14]:
customers.write.format("mongodb").option("database", "shopdb").option("collection", "purchases").mode("overwrite").save()

In [15]:
vip_clients = purchases.groupBy("customer_id").agg(_sum("cost").alias("total_spent")).orderBy(col("total_spent").desc()).limit(5)
vip_clients.show(5)

+------------+-----------+
| customer_id|total_spent|
+------------+-----------+
|zk8s4xalhygb|      15642|
|z5p6aqq2de27|      13995|
|6wu2vfucnr62|      13257|
|afldym6j2i2s|      11874|
|hg7stpi94f2y|      11661|
+------------+-----------+

